In [37]:
import numpy as np
import pandas as pd
import random
#from faker import Faker
from ds_collection import AVLTreeMap
import csv
from collections import defaultdict, deque

### Data Generation

In [7]:
# fake = Faker()

# N = 1_000_000
# df = pd.DataFrame({
#     "incident_id": np.arange(1, N + 1),
#     "year": np.random.randint(2010, 2026, size=N),
#     "city": [fake.city() for _ in range(N)],
#     "state": [fake.state() for _ in range(N)],
#     "victim_age": np.random.randint(10, 91, size=N),
#     "offender_age": np.random.randint(12, 91, size=N),
#     "weapon_used": np.random.choice(["Firearm", "Knife", "Blunt Object", "Strangulation", "Poison", "Fire", "Vehicle", "Explosive", "Unknown"], size=N),
#     "relationship": np.random.choice(["Stranger", "Acquaintance", "Spouse", "Parent", "Child", "Sibling", "Friend", "Neighbor", "Coworker", "Unknown"], size=N),
#     "crime_category": np.random.choice(["Homicide", "Assault", "Robbery", "Manslaughter", "Domestic Violence"], size=N)

# })
# df.to_csv("crime_reports_fast.csv", index=False)

### Raw data

In [38]:
data = {}
with open("crime_reports_fast.csv", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        row["year"] = int(row["year"])
        row["incident_id"] = int(row["incident_id"])
        row["victim_age"] = int(row["victim_age"])
        row["offender_age"] = int(row["offender_age"])
        idd = row["incident_id"]
        data[idd] = row
data


{3: {'incident_id': 3,
  'year': 2017,
  'city': 'New Luisborough',
  'state': 'Caroline ',
  'victim_age': 78,
  'offender_age': 75,
  'weapon_used': 'gun',
  'relationship': 'Coworker',
  'crime_category': 'Robbery'},
 4: {'incident_id': 4,
  'year': 2022,
  'city': 'Port Tiffanyshire',
  'state': 'Pennsylvania',
  'victim_age': 87,
  'offender_age': 78,
  'weapon_used': 'Fire',
  'relationship': 'Acquaintance',
  'crime_category': 'Robbery'},
 5: {'incident_id': 5,
  'year': 2015,
  'city': 'Kevinport',
  'state': 'Georgia',
  'victim_age': 73,
  'offender_age': 47,
  'weapon_used': 'Poison',
  'relationship': 'Stranger',
  'crime_category': 'Homicide'},
 6: {'incident_id': 6,
  'year': 2013,
  'city': 'North Jennifer',
  'state': 'Massachusetts',
  'victim_age': 35,
  'offender_age': 56,
  'weapon_used': 'Explosive',
  'relationship': 'Sibling',
  'crime_category': 'Assault'},
 7: {'incident_id': 7,
  'year': 2010,
  'city': 'Robertton',
  'state': 'Indiana',
  'victim_age': 82,
  

# DATA STRUCTURES 

In [39]:
class ColumnsAVL(AVLTreeMap):
    def put(self, k, v):
        self._check_key(k)
        p = self._subtree_search(self._tree.root(), k)

        if self._tree.is_external(p):
            # Key not found: expand external and rebalance
            self._expand_external(p, self._MapEntry(k, [int(v)]))
            self._rebalance_insert(p)
            return None
        else:
            # Add new value to the same key
            value = p.get_element().get_value()
            value.append(int(v))
            return None

    def remove(self, k, v):
        if self.is_empty():
            return None

        p = self._subtree_search(self._tree.root(), k)
        if self._tree.is_external(p):
            # Key not found
            return None

        old_value = p.get_element().get_value()
        if len(old_value) == 1:

            # Case: two internal children → replace with in-order predecessor
            if self._tree.is_internal(self._tree.left(p)) and self._tree.is_internal(self._tree.right(p)):
                replacement = self._tree_max(self._tree.left(p))
                self._tree.set(p, replacement.get_element())
                p = replacement

            # Now p has at most one internal child
            child_sentinel = self._tree.left(p) if self._tree.is_external(self._tree.left(p)) else self._tree.right(p)
            sibling = self._tree.sibling(child_sentinel)

            self._tree.remove(child_sentinel)
            self._tree.remove(p)
            self._rebalance_delete(sibling)

            return old_value[0]
        else:
            value = p.get_element().get_value()
            value.remove(v)
            return v

# 1. Year column

In [40]:
year_column = ColumnsAVL()
for incident_id, row in data.items():
    year_column.put(row["year"], incident_id)
len(year_column)

18

# 2. City column

In [41]:
city_column = ColumnsAVL()
for incident_id, row in data.items():
    city_column.put(row["city"], incident_id)
len(city_column)

91466

# 3. State column

In [42]:
state_column = ColumnsAVL()
for incident_id, row in data.items():
    state_column.put(row["state"], incident_id)
len(state_column)

53

# 4. Victim Age column

In [43]:
victim_age = ColumnsAVL()
for incident_id, row in data.items():
    victim_age.put(row["victim_age"], incident_id)
len(victim_age)

82

# 5. Offender age column


In [44]:
offender_age = ColumnsAVL()
for incident_id, row in data.items():
    offender_age.put(row["offender_age"], incident_id)
len(offender_age)

80

# 6. Weapon Used column


In [45]:
weapon_used = ColumnsAVL()
for incident_id, row in data.items():
    weapon_used.put(row["weapon_used"], incident_id)
len(weapon_used)

12

# 7. Relationship column


In [48]:
relationship_column = ColumnsAVL()
for incident_id, row in data.items():
    relationship_column.put(row["relationship"], incident_id)
len(relationship_column)

11

# 8. Crime_category column

In [49]:
crime_category_column = ColumnsAVL()
for incident_id, row in data.items():
    crime_category_column.put(row["crime_category"], incident_id)
len(crime_category_column)

6

# Graphs 


In [50]:
class AdjacencyMapGraph:
    def __init__(self, directed=True):
        self.directed = directed
        self.graph = {}  # {u: {v: weight}}

    def add_vertex(self, v):
        if v not in self.graph:
            self.graph[v] = {}

    def add_edge(self, u, v, w=1):
        self.add_vertex(u)
        self.add_vertex(v)
        self.graph[u][v] = w
        if not self.directed:
            self.graph[v][u] = w

    def vertices(self):
        return list(self.graph.keys())

    def edges(self):
        e = []
        for u in self.graph:
            for v, w in self.graph[u].items():
                e.append((u, v, w))
        return e

    def neighbors(self, v):
        return list(self.graph[v].keys())


In [ ]:
from collections import defaultdict

def build_city_similarity_graph(data):
    combo_counts = defaultdict(lambda: defaultdict(int))
    graph = AdjacencyMapGraph(directed=False)

    for rec in data.values():
        state = rec["state"]
        category = rec["crime_category"]
        weapon = rec["weapon_used"]
        combo_counts[(category, weapon)][state] += 1

    for (category, weapon), state_freq in combo_counts.items():
        states = list(state_freq.keys())
        for i in range(len(states)):
            for j in range(i + 1, len(states)):
                state_a, state_b = states[i], states[j]
                weight = max(state_freq[state_a], state_freq[state_b])
                current = graph.graph.get(state_a, {}).get(state_b, 0)
                graph.add_edge(state_a, state_b, current + weight)

    return graph



In [52]:
_city_graph_display = {}
_graph_instance = AdjacencyMapGraph(directed=False)

In [ ]:
def _compute_graph_structures(records):
    combo_counts = defaultdict(lambda: defaultdict(int))
    for rec in records.values():
        state = rec["state"]
        category = rec["crime_category"]
        weapon = rec["weapon_used"]
        combo_counts[(category, weapon)][state] += 1

    display = defaultdict(lambda: defaultdict(list))
    graph_obj = AdjacencyMapGraph(directed=False)

    for (category, weapon), state_freq in combo_counts.items():
        states = list(state_freq.keys())
        for i in range(len(states)):
            for j in range(i + 1, len(states)):
                state_a, state_b = states[i], states[j]
                weight = state_freq[state_a] + state_freq[state_b]
                display[state_a][state_b].append(
                    {"crime": category, "weapon": weapon, "weight": weight}
                )
                display[state_b][state_a].append(
                    {"crime": category, "weapon": weapon, "weight": weight}
                )
                current = graph_obj.graph.get(state_a, {}).get(state_b, 0)
                graph_obj.add_edge(state_a, state_b, current + weight)

    normalized = {state: dict(neighbors) for state, neighbors in display.items()}
    return normalized, graph_obj


In [58]:
trees = {"year": year_column, "city": city_column, "state": state_column, "victim_age": victim_age,
         "offender_age": offender_age, "weapon_used": weapon_used, "relationship": relationship_column,
          "crime_category": crime_category_column}


In [59]:
def _rebuild_graph_structures():
    global _city_graph_display, _graph_instance
    _city_graph_display, _graph_instance = _compute_graph_structures(data)

_rebuild_graph_structures()

# Search

In [ ]:
def search(trees, column, key, incident_id=None):
    tree = trees[column]
    values = tree.get(key)
    if not values:
        return [], None

    record = None
    if incident_id is not None and incident_id in values:
        record = data.get(incident_id)

    return values, record


In [61]:
search(trees, "year", 2017)

([3,
  20,
  24,
  38,
  70,
  74,
  78,
  94,
  111,
  116,
  125,
  147,
  184,
  202,
  216,
  217,
  227,
  245,
  251,
  255,
  260,
  263,
  284,
  287,
  306,
  308,
  315,
  342,
  384,
  413,
  418,
  439,
  443,
  446,
  464,
  469,
  476,
  491,
  506,
  512,
  517,
  528,
  547,
  554,
  556,
  557,
  581,
  584,
  652,
  653,
  656,
  657,
  693,
  734,
  740,
  742,
  753,
  774,
  792,
  808,
  843,
  851,
  868,
  902,
  919,
  942,
  957,
  996,
  1003,
  1010,
  1020,
  1034,
  1061,
  1083,
  1088,
  1096,
  1141,
  1190,
  1192,
  1213,
  1283,
  1297,
  1333,
  1342,
  1359,
  1362,
  1377,
  1388,
  1405,
  1416,
  1438,
  1444,
  1449,
  1457,
  1484,
  1547,
  1550,
  1553,
  1622,
  1632,
  1656,
  1670,
  1681,
  1709,
  1714,
  1735,
  1737,
  1752,
  1754,
  1763,
  1774,
  1777,
  1784,
  1792,
  1814,
  1837,
  1857,
  1865,
  1885,
  1893,
  1898,
  1908,
  1949,
  1959,
  2011,
  2022,
  2031,
  2052,
  2055,
  2073,
  2094,
  2112,
  2115,
  2116,
  213

### Multisearch


In [ ]:
def multisearch(trees, columns, keys):
    candidate_sets = []
    for column, key in zip(columns, keys):
        tree = trees[column]
        values = tree.get(key)
        if not values:
            return f"There are no such incidents where the {column} is {key}."
        candidate_sets.append(set(values))

    commons = set.intersection(*candidate_sets) if candidate_sets else set()
    if not commons:
        return "There are no such incidents where the given criteria satisfy."

    results = {incident_id: data[incident_id] for incident_id in sorted(commons)}
    return results

In [63]:
multisearch(trees, ["year", "state"], [2017, "Wisconsin"])

{2656: {'incident_id': 2656,
  'year': 2017,
  'city': 'Nicoleville',
  'state': 'Wisconsin',
  'victim_age': 25,
  'offender_age': 14,
  'weapon_used': 'Fire',
  'relationship': 'Coworker',
  'crime_category': 'Homicide'},
 2660: {'incident_id': 2660,
  'year': 2017,
  'city': 'Port Brad',
  'state': 'Wisconsin',
  'victim_age': 17,
  'offender_age': 21,
  'weapon_used': 'Firearm',
  'relationship': 'Unknown',
  'crime_category': 'Domestic Violence'},
 3102: {'incident_id': 3102,
  'year': 2017,
  'city': 'Mcmillanshire',
  'state': 'Wisconsin',
  'victim_age': 87,
  'offender_age': 67,
  'weapon_used': 'Vehicle',
  'relationship': 'Acquaintance',
  'crime_category': 'Robbery'},
 3680: {'incident_id': 3680,
  'year': 2017,
  'city': 'Jessicaborough',
  'state': 'Wisconsin',
  'victim_age': 85,
  'offender_age': 22,
  'weapon_used': 'Explosive',
  'relationship': 'Parent',
  'crime_category': 'Homicide'},
 3697: {'incident_id': 3697,
  'year': 2017,
  'city': 'Berrymouth',
  'state': '

# Range Queries

In [ ]:
NUMERIC_RANGE_COLUMNS = {"year", "victim_age", "offender_age"}
def range_query(column, start, end):
    tree = trees[column]
    results = {}

    if column in NUMERIC_RANGE_COLUMNS:
        start_val = int(start)
        end_val = int(end)
        upper_bound = end_val + 1
    else:
        start_val = str(start)
        end_val = str(end)
        upper_bound = end_val

    if start_val > end_val:
        start_val, end_val = end_val, start_val
        if column in NUMERIC_RANGE_COLUMNS:
            upper_bound = end_val + 1
        else:
            upper_bound = end_val

    seen_keys = set()
    for entry in tree.sub_map(start_val, upper_bound):
        key = entry.get_key()
        seen_keys.add(key)
        for incident_id in entry.get_value():
            record = data.get(incident_id)
            if record:
                results[incident_id] = record

    if column not in NUMERIC_RANGE_COLUMNS and end_val not in seen_keys:
        ids = tree.get(end_val)
        if ids:
            for incident_id in ids:
                record = data.get(incident_id)
                if record:
                    results[incident_id] = record

    return results

In [65]:
range_query("victim_age", 24, 25)

{76: {'incident_id': 76,
  'year': 2016,
  'city': 'Lake Amybury',
  'state': 'Mississippi',
  'victim_age': 24,
  'offender_age': 82,
  'weapon_used': 'Knife',
  'relationship': 'Parent',
  'crime_category': 'Assault'},
 226: {'incident_id': 226,
  'year': 2016,
  'city': 'Shermanberg',
  'state': 'New Jersey',
  'victim_age': 24,
  'offender_age': 50,
  'weapon_used': 'Knife',
  'relationship': 'Neighbor',
  'crime_category': 'Robbery'},
 250: {'incident_id': 250,
  'year': 2015,
  'city': 'Port Emilyview',
  'state': 'Tennessee',
  'victim_age': 24,
  'offender_age': 66,
  'weapon_used': 'Unknown',
  'relationship': 'Neighbor',
  'crime_category': 'Manslaughter'},
 318: {'incident_id': 318,
  'year': 2020,
  'city': 'North Michael',
  'state': 'North Carolina',
  'victim_age': 24,
  'offender_age': 61,
  'weapon_used': 'Blunt Object',
  'relationship': 'Friend',
  'crime_category': 'Homicide'},
 536: {'incident_id': 536,
  'year': 2020,
  'city': 'Juliemouth',
  'state': 'Maine',
  

# Deletion


In [ ]:
def delete_record(incident_id: int, csv_path: str = "crime_reports_fast.csv") -> bool:
    global data, trees

    incident = data.get(incident_id)
    if not incident:
        print(f"Incident {incident_id} not found.")
        return False

    fieldnames_template = list(incident.keys())

    def remove_from_index(index, key, id_value):
        ids = index.get(key)
        if ids is None:
            return
        index.remove(key, id_value)

    for column, value in incident.items():
        if column in trees:
            remove_from_index(trees[column], value, incident_id)

    data.pop(incident_id)
    _rebuild_graph_structures()

    if csv_path:
        if data:
            fieldnames = list(next(iter(data.values())).keys())
        else:
            fieldnames = fieldnames_template
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data.values())

    print(f"Incident {incident_id} successfully deleted and saved to {csv_path}.")
    return True



testing deletion

In [67]:
delete_record(1)

Incident 1 not found.


False

# Insertion

In [68]:
def insert_record(year="None", city="None", state="None", victim_age=0, offender_age=0, weapon_used="None", relationship="None", crime_category="None", csv_path="crime_reports_fast.csv"):
    global data, trees
    new_id = max(data.keys()) + 1
    data[new_id] = {
        "incident_id": new_id,
        "year": year,
        "city": city,
        "state": state,
        "victim_age": victim_age,
        "offender_age": offender_age,
        "weapon_used": weapon_used,
        "relationship": relationship,
        "crime_category": crime_category
    }

    for key, value in data[new_id].items():
        if key in trees:
            trees[key].put(value, new_id)

    if csv_path:
        fieldnames = ["incident_id", "year", "city", "state", "victim_age", "offender_age", "weapon_used", "relationship", "crime_category"]
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if f.tell() == 0:
                writer.writeheader()
            writer.writerow(data[new_id])

    _rebuild_graph_structures()
    return new_id


In [69]:
id1 = insert_record(
    year=2020,
    city="Los Angeles",
    state="CA",
    victim_age=25,
    offender_age=30,
    weapon_used="Handgun",
    relationship="Stranger",
    crime_category="Homicide"
)

id2 = insert_record(
    year=2021,
    city="Houston",
    state="TX",
    victim_age=17,
    offender_age=18,
    weapon_used="Knife",
    relationship="Friend",
    crime_category="Assault"
)

print("Inserted IDs:", id1, id2)
print(data[id1-1], data[id2-1])

Inserted IDs: 1000005 1000006
{'incident_id': 1000004, 'year': 0, 'city': 'None', 'state': 'None', 'victim_age': 0, 'offender_age': 0, 'weapon_used': 'None', 'relationship': 'None', 'crime_category': 'None'} {'incident_id': 1000005, 'year': 2020, 'city': 'Los Angeles', 'state': 'CA', 'victim_age': 25, 'offender_age': 30, 'weapon_used': 'Handgun', 'relationship': 'Stranger', 'crime_category': 'Homicide'}


# Modification

In [ ]:
def modify_record(incident_id, csv_path="crime_reports_fast.csv", **fields):
    global trees, data

    record = data.get(incident_id)
    if record is None:
        print(f"Incident {incident_id} not found.")
        return None

    updated_keys = []
    for key, val in fields.items():
        if key not in record:
            continue
        old = record[key]
        if old == val:
            continue
        record[key] = val
        if key in trees:
            tree = trees[key]
            tree.remove(old, incident_id)
            tree.put(val, incident_id)
        updated_keys.append(key)

    if not updated_keys:
        return record

    if csv_path:
        fieldnames = ["incident_id", "year", "city", "state", "victim_age", "offender_age", "weapon_used", "relationship", "crime_category"]
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for incident in sorted(data.values(), key=lambda row: row["incident_id"]):
                writer.writerow(incident)

    _rebuild_graph_structures()
    return record


In [73]:
modify_record(1000001, **{'state':"Arizona"})

{'incident_id': 1000001,
 'year': 2020,
 'city': 'Los Angeles',
 'state': 'Arizona',
 'victim_age': 25,
 'offender_age': 30,
 'weapon_used': 'Handgun',
 'relationship': 'Stranger',
 'crime_category': 'Homicide'}

In [74]:
graph = AdjacencyMapGraph()
build_city_similarity_graph(data)

In [78]:
def get_city_graph():
    return _city_graph_display

In [98]:
def graph(metric="sum"):
    return _graph_instance


## DFS

In [79]:
def dfs(graph, start):
    visited = set()
    order = []

    def explore(v):
        visited.add(v)
        order.append(v)
        for nbr in graph.graph[v]:
            if nbr not in visited:
                explore(nbr)

    explore(start)
    return order

## BFS

In [80]:
def bfs(graph, start):
    visited = set([start])
    dist = {start: 0}
    queue = deque([start])

    while queue:
        u = queue.popleft()
        for v in graph.graph[u]:
            if v not in visited:
                visited.add(v)
                dist[v] = dist[u] + 1
                queue.append(v)

    return dist


## Bellman's algorithm

In [ ]:
def bellman_ford(graph, source):
    dist = {v: float("inf") for v in graph.graph}
    parent = {v: None for v in graph.graph}
    dist[source] = 0

    edges = []
    for u in graph.graph:
        for v, w in graph.graph[u].items():
            edges.append((u, v, w))

    V = len(graph.graph)

    for _ in range(V - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u

    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            print("Negative cycle detected!")

    return dist, parent


In [82]:
graph = build_city_similarity_graph(data)

print("Cities:", graph.vertices()[:10])
print("Edges:", graph.edges()[:10])

start_city = graph.vertices()[0]

print("DFS:", dfs(graph, start_city))
print("BFS:", bfs(graph, start_city))

dist, parent = bellman_ford(graph, start_city)
print("Bellman-Ford distances:", dist)


Cities: ['Pennsylvania', 'Arkansas', 'Connecticut', 'Nevada', 'Indiana', 'New Jersey', 'Wisconsin', 'Oklahoma', 'Maryland', 'New Hampshire']
Edges: [('Pennsylvania', 'Arkansas', 20502), ('Pennsylvania', 'Connecticut', 20597), ('Pennsylvania', 'Nevada', 20611), ('Pennsylvania', 'Indiana', 20625), ('Pennsylvania', 'New Jersey', 20486), ('Pennsylvania', 'Wisconsin', 20497), ('Pennsylvania', 'Oklahoma', 20685), ('Pennsylvania', 'Maryland', 20454), ('Pennsylvania', 'New Hampshire', 20571), ('Pennsylvania', 'Illinois', 20504)]
DFS: ['Pennsylvania', 'Arkansas', 'Connecticut', 'Nevada', 'Indiana', 'New Jersey', 'Wisconsin', 'Oklahoma', 'Maryland', 'New Hampshire', 'Illinois', 'Missouri', 'Hawaii', 'Massachusetts', 'North Dakota', 'Delaware', 'New Mexico', 'Texas', 'Minnesota', 'New York', 'Nebraska', 'Alaska', 'Oregon', 'Montana', 'Maine', 'Florida', 'North Carolina', 'Vermont', 'Iowa', 'Kansas', 'South Carolina', 'Louisiana', 'Virginia', 'Tennessee', 'Kentucky', 'Utah', 'Idaho', 'Rhode Island

In [83]:
start = graph.vertices()[0]
print("DFS:", dfs(graph, start))
print("BFS:", bfs(graph, start))
dist, parent = bellman_ford(graph, start)
print("Shortest paths example:", list(dist.items())[:20])


DFS: ['Pennsylvania', 'Arkansas', 'Connecticut', 'Nevada', 'Indiana', 'New Jersey', 'Wisconsin', 'Oklahoma', 'Maryland', 'New Hampshire', 'Illinois', 'Missouri', 'Hawaii', 'Massachusetts', 'North Dakota', 'Delaware', 'New Mexico', 'Texas', 'Minnesota', 'New York', 'Nebraska', 'Alaska', 'Oregon', 'Montana', 'Maine', 'Florida', 'North Carolina', 'Vermont', 'Iowa', 'Kansas', 'South Carolina', 'Louisiana', 'Virginia', 'Tennessee', 'Kentucky', 'Utah', 'Idaho', 'Rhode Island', 'Mississippi', 'California', 'Arizona', 'Wyoming', 'Georgia', 'West Virginia', 'Colorado', 'Washington', 'South Dakota', 'Alabama', 'Michigan', 'Ohio', 'TX', 'CA']
BFS: {'Pennsylvania': 0, 'Arkansas': 1, 'Connecticut': 1, 'Nevada': 1, 'Indiana': 1, 'New Jersey': 1, 'Wisconsin': 1, 'Oklahoma': 1, 'Maryland': 1, 'New Hampshire': 1, 'Illinois': 1, 'Missouri': 1, 'Hawaii': 1, 'Massachusetts': 1, 'North Dakota': 1, 'Delaware': 1, 'New Mexico': 1, 'Texas': 1, 'Minnesota': 1, 'New York': 1, 'Nebraska': 1, 'Alaska': 1, 'Oregon

# Analytics and Statistics 

In [84]:
import statistics

def get_column_values(data, col):
    """Extract non-null numeric values for a given column."""
    values = []
    for rec in data.values():
        if col in rec and rec[col] != "" and rec[col] is not None:
            try:
                values.append(float(rec[col]))
            except:
                pass
    return values


In [87]:
def compute_min(data, col):
    values = get_column_values(data, col)
    return min(values) if values else None

def compute_max(data, col):
    values = get_column_values(data, col)
    return max(values) if values else None

def compute_average(data, col):
    values = get_column_values(data, col)
    return sum(values) / len(values) if values else None

def compute_median(data, col):
    values = get_column_values(data, col)
    return statistics.median(values) if values else None


In [88]:
print("Min victim age:", compute_min(data, "victim_age"))
print("Max victim age:", compute_max(data, "victim_age"))
print("Average victim age:", compute_average(data, "victim_age"))
print("Median victim age:", compute_median(data, "victim_age"))


Min victim age: 0.0
Max victim age: 90.0
Average victim age: 50.03907088278735
Median victim age: 50.0


In [89]:
from collections import Counter

def top_k_frequent(data, col, K=10):
    counter = Counter()

    for rec in data.values():
        if col in rec:
            counter[rec[col]] += 1

    return counter.most_common(K)

In [90]:
print("Top 10 crime categories:", top_k_frequent(data, "crime_category", K=10))
print("Top 10 cities:", top_k_frequent(data, "city", K=10))


Top 10 crime categories: [('Domestic Violence', 200427), ('Manslaughter', 200270), ('Homicide', 200093), ('Robbery', 199947), ('Assault', 199264), ('None', 2)]
Top 10 cities: [('New Michael', 896), ('South Michael', 891), ('West Michael', 838), ('North Michael', 818), ('Port Michael', 803), ('Lake Michael', 795), ('East Michael', 749), ('Michaelmouth', 595), ('New David', 589), ('Smithmouth', 582)]


In [91]:
def top_k_by_value(data, col, K=10):
    records = []

    for rec in data.values():
        if col in rec:
            try:
                records.append((rec[col], rec))
            except:
                pass

    # Sort by numeric value descending
    records.sort(key=lambda x: float(x[0]), reverse=True)
    return records[:K]


In [92]:
print("Top 5 highest victim ages:", top_k_by_value(data, "victim_age", 5))


Top 5 highest victim ages: [(90, {'incident_id': 24, 'year': 2017, 'city': 'Donaldtown', 'state': 'Hawaii', 'victim_age': 90, 'offender_age': 43, 'weapon_used': 'Unknown', 'relationship': 'Child', 'crime_category': 'Manslaughter'}), (90, {'incident_id': 132, 'year': 2014, 'city': 'Katherinetown', 'state': 'Delaware', 'victim_age': 90, 'offender_age': 16, 'weapon_used': 'Poison', 'relationship': 'Stranger', 'crime_category': 'Manslaughter'}), (90, {'incident_id': 342, 'year': 2017, 'city': 'Danielmouth', 'state': 'Illinois', 'victim_age': 90, 'offender_age': 17, 'weapon_used': 'Poison', 'relationship': 'Child', 'crime_category': 'Homicide'}), (90, {'incident_id': 519, 'year': 2012, 'city': 'Stewartton', 'state': 'Massachusetts', 'victim_age': 90, 'offender_age': 47, 'weapon_used': 'Fire', 'relationship': 'Unknown', 'crime_category': 'Robbery'}), (90, {'incident_id': 537, 'year': 2010, 'city': 'Port Ashley', 'state': 'Maryland', 'victim_age': 90, 'offender_age': 16, 'weapon_used': 'Knife

# KNN 

In [93]:
#KNN
def record_to_vector(rec, features):
    vec = []
    for f in features:
        try:
            vec.append(float(rec[f]))
        except:
            vec.append(0.0)
    return vec


In [ ]:
import math
def euclidean(v1, v2):
    return math.sqrt(sum((a - b)**2 for a, b in zip(v1, v2)))


In [ ]:
def knn(data, query_id, features, K=5):
    target = data.get(query_id)
    if target is None:
        return []

    target_vec = record_to_vector(target, features)
    distances = []
    for rid, rec in data.items():
        if rid == query_id:
            continue
        vec = record_to_vector(rec, features)
        dist = euclidean(target_vec, vec)
        distances.append((dist, rid, rec))

    distances.sort(key=lambda x: x[0])
    top = []
    for dist, rid, rec in distances[:K]:
        top.append({
            "incident_id": rid,
            "distance": dist,
            "record": rec,
        })
    return top

